# AgentCore Evaluations Lab
## Mastering Amazon Bedrock AgentCore | Pumping Code

---

## 🎯 What You'll Build

In this lab you will set up a **complete evaluation pipeline** for a Strands AI agent using **Amazon Bedrock AgentCore Evaluations**.

By the end of this lab, you will have:
- ✅ Deployed a Strands agent to AgentCore Runtime with automatic OTEL instrumentation
- ✅ Explored all 13 built-in evaluators
- ✅ Created a custom LLM-as-judge evaluator with a 5-level scoring scale
- ✅ Run **on-demand evaluations** at session, trace, and span levels
- ✅ Interpreted evaluation results (label, value, explanation)
- ✅ Configured **online evaluation** for continuous production monitoring
- ✅ Saved evaluation results to a JSON file

---

## 🏗️ Architecture

```
┌──────────────────────────────────────────────┐
│           Strands Agent                      │
│   (Claude Haiku 4.5 + Math + Weather tools)  │
└──────────────┬───────────────────────────────┘
               │  Deployed to AgentCore Runtime
               ▼
┌──────────────────────────────────────────────┐
│      AgentCore Runtime                       │
│  (auto OTEL via AgentCore Runtime)    │
└──────────────┬───────────────────────────────┘
               │  OTEL Traces
               ▼
┌──────────────────────────────────────────────┐
│    AgentCore Observability + CloudWatch      │
│  Sessions → Traces → Spans (Tool Calls)      │
└──────────────┬───────────────────────────────┘
               │
      ┌────────┴────────┐
      ▼                 ▼
 On-Demand          Online Eval
 Evaluation         Configuration
 (developer         (continuous
  triggered)         sampling)
```

---


## ✅ Prerequisites
- AWS CLI configured with appropriate credentials
- Python 3.10+
- Access to Amazon Bedrock
- Bedrock access enabled for: `us.anthropic.claude-haiku-4-5-20251001-v1:0`
- Access to ECR (for container deployment)

This lab runs TypeScript on the Deno kernel. Pick the **Deno** kernel in the top right.

---
# Part 1: Environment Setup

In [ ]:
// Dependencies are pinned in the project's deno.json and cached by ./setup.sh; there is no install
// step. (The Python lab installed bedrock-agentcore, the starter toolkit, boto3, pickleshare,
// strands-agents and strands-agents-tools here.)
import { sh } from "../shared/notebook.ts";
await sh("deno", ["--version"]);

In [ ]:
Deno.env.set("AWS_REGION", "us-east-1");

// APPROACH A: Use credentials
// Deno.env.set("AWS_ACCESS_KEY_ID", "your_access_key");
// Deno.env.set("AWS_SECRET_ACCESS_KEY", "your_secret_key");
// Deno.env.set("AWS_SESSION_TOKEN", "your_session_token");

// APPROACH B: Use AWS SSO profile
// Deno.env.set("AWS_PROFILE", "your_profile");

// Remove any existing credential env vars to force profile usage
// for (const key of ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]) {
//   Deno.env.delete(key);
// }

Deno.env.set("AWS_REGION", "us-east-1");

console.log("✅ AWS Profile set. Please restart kernel and run all cells.");

In [ ]:
import { GetCallerIdentityCommand, STSClient } from "@aws-sdk/client-sts";
import { loadEnv, state, writeFile } from "../shared/notebook.ts";

await loadEnv();

const region = Deno.env.get("AWS_REGION") ?? "us-east-1";

try {
  const identity = await new STSClient({ region }).send(new GetCallerIdentityCommand({}));
  console.log("✅ AWS Credentials Verified");
  console.log(`   Account: ${identity.Account}`);
  console.log(`   ARN:     ${identity.Arn}`);
  console.log(`   Region:  ${region}`);
} catch (e) {
  console.log(`❌ AWS Credentials Error: ${e}`);
}

---
# Part 2: Deploy a Strands Agent to AgentCore Runtime

We'll deploy a simple **weather + math assistant** to AgentCore Runtime.

This agent is intentionally simple so we can test evaluation metrics effectively:
- ✅ Math questions → should answer correctly
- ✅ Weather questions → should answer correctly (using weather tool)
- ❌ Out-of-scope questions → our custom evaluator should penalize these

> **Important**: AgentCore Runtime already emits OTEL traces for deployed agents — no extra OTEL package is required here.

In [ ]:
// Write the agent entry point file. `writeFile` resolves relative paths against the notebooks
// directory, so the Python lab's `get_project_root()` walk is not needed here.
const EVALUATION_DIR = "../backend/evaluation";
const AGENT_FILE = `${EVALUATION_DIR}/eval_agent_strands.ts`;
// The agent's own deno.json plays the role of the Python lab's requirements_eval.txt.
const REQUIREMENTS_FILE = `${EVALUATION_DIR}/deno.json`;

const AGENT_CODE = `import { Agent, BedrockModel, tool } from "@strands-agents/sdk";
import { BedrockAgentCoreApp } from "bedrock-agentcore/runtime";
import { z } from "zod";

const MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0";

// Strands TypeScript ships no \`calculator\`, so the lab brings its own - the same tool the
// observability lab (notebook 09) generates, so both labs exercise identical agent behaviour.
const calculator = tool({
  name: "calculator",
  description: "Evaluate an arithmetic expression, e.g. '18 * 7'.",
  inputSchema: z.object({ expression: z.string() }),
  callback: ({ expression }) => {
    if (!/^[\\d\\s+\\-*/().]+$/.test(expression)) return "invalid expression";
    return String(Function(\`"use strict"; return (\${expression});\`)());
  },
});

const getWeather = tool({
  name: "get_weather",
  description: "Get current weather for a location. Returns a mock response for this demo.",
  inputSchema: z.object({ location: z.string() }),
  // Simplified mock - real implementation would call a weather API
  callback: ({ location }) => ({
    location,
    temperature: "22°C",
    condition: "Partly cloudy",
    humidity: "65%",
  }),
});

const model = new BedrockModel({ modelId: MODEL_ID });

const agent = new Agent({
  model,
  tools: [calculator, getWeather],
  systemPrompt: "You are a helpful assistant. You can perform math calculations " +
    "and check the weather. Stay focused on these topics only.",
});

const app = new BedrockAgentCoreApp({
  invocationHandler: {
    requestSchema: z.object({ prompt: z.string().default("") }),
    process: async ({ prompt }) => (await agent.invoke(prompt)).toString(),
  },
});

app.run();
`;

interface RootConfig {
  imports: Record<string, string>;
}

const rootImports = (JSON.parse(await Deno.readTextFile("../../deno.json")) as RootConfig).imports;
const REQUIREMENTS = JSON.stringify(
  {
    nodeModulesDir: "auto",
    compilerOptions: { strict: true },
    imports: Object.fromEntries(
      [
        "@strands-agents/sdk",
        "bedrock-agentcore/",
        "zod",
        "@opentelemetry/api",
        "@opentelemetry/api-logs",
        "@opentelemetry/context-async-hooks",
        "@opentelemetry/core",
        "@opentelemetry/otlp-transformer",
        "@opentelemetry/resources",
        "@opentelemetry/sdk-logs",
        "@opentelemetry/sdk-trace-base",
        "@smithy/protocol-http",
        "@smithy/signature-v4",
        "@aws-crypto/sha256-js",
        "@aws-sdk/credential-provider-node",
      ].map((k) => [k, rootImports[k]]),
    ),
  },
  null,
  2,
) + "\n";

// Save to files for deployment
await writeFile(AGENT_FILE, AGENT_CODE);
await writeFile(REQUIREMENTS_FILE, REQUIREMENTS);

console.log(`✅ Agent code written to: ${AGENT_FILE}`);
console.log(`✅ Requirements written to: ${REQUIREMENTS_FILE}`);
console.log();
console.log("Agent capabilities:");
console.log("  🧮 calculator — math operations");
console.log("  🌤️  get_weather — weather lookups");
console.log("  ❌ Out-of-scope — custom evaluator will penalize these");

In [ ]:
import {
  BedrockAgentCoreControlClient,
  ListAgentRuntimesCommand,
} from "@aws-sdk/client-bedrock-agentcore-control";
import { Runtime } from "../toolkit/mod.ts";
import { CONFIG_FILE as RUNTIME_CONFIG_FILE } from "../toolkit/runtime.ts";

const RUNTIME_WORK_DIR = EVALUATION_DIR;

const ctrl = new BedrockAgentCoreControlClient({ region });
const sleep = (ms: number) => new Promise((resolve) => setTimeout(resolve, ms));

/** Wait until no runtime with this logical name is stuck in DELETING. */
async function waitForRuntimeNameToClear(
  agentName: string,
  attempts = 20,
  delaySeconds = 15,
): Promise<void> {
  for (let attempt = 1; attempt <= attempts; attempt++) {
    const listed = await ctrl.send(new ListAgentRuntimesCommand({ maxResults: 100 }));
    const matching = (listed.agentRuntimes ?? []).filter((item) =>
      item.agentRuntimeName === agentName
    );
    const deleting = matching.filter((item) => item.status === "DELETING");
    if (deleting.length === 0) return;
    const ids = deleting.map((item) => item.agentRuntimeId ?? "?").join(", ");
    if (attempt === attempts) {
      throw new Error(`Runtime name ${agentName} is still deleting after waiting: ${ids}`);
    }
    console.log(`⏳ Existing runtime still deleting (${ids}). Waiting ${delaySeconds}s...`);
    await sleep(delaySeconds * 1000);
  }
}

// The Python lab deletes a stale `.bedrock_agentcore.yaml` so deployment starts cleanly. The
// toolkit's config is JSON, keyed by agent name and shared with the other labs, so only this
// agent's entry goes: a stale agentId would otherwise make launch() update a deleted runtime.
async function clearStoredRuntimeConfig(agentName: string): Promise<void> {
  try {
    const all = JSON.parse(await Deno.readTextFile(RUNTIME_CONFIG_FILE)) as Record<string, unknown>;
    if (!(agentName in all)) return;
    delete all[agentName];
    await Deno.writeTextFile(RUNTIME_CONFIG_FILE, `${JSON.stringify(all, null, 2)}\n`);
    console.log(`🧹 Removed stale ${agentName} entry from ${RUNTIME_CONFIG_FILE}`);
  } catch {
    // Nothing stored yet.
  }
}

const AGENT_NAME = "eval_lab_agent";

await clearStoredRuntimeConfig(AGENT_NAME);

const agentcoreRuntime = new Runtime();

await waitForRuntimeNameToClear(AGENT_NAME);

console.log(`📁 Runtime work directory: ${RUNTIME_WORK_DIR}`);

console.log(`🚀 Deploying agent: ${AGENT_NAME}`);
console.log("   This builds a Docker container and deploys to AgentCore Runtime.");
console.log("   ⏱️  This takes ~10-15 minutes. Continue reading while it runs!\n");

await agentcoreRuntime.configure({
  entrypoint: "eval_agent_strands.ts",
  autoCreateExecutionRole: true,
  autoCreateEcr: true,
  requirementsFile: "deno.json",
  region,
  agentName: AGENT_NAME,
  idleTimeout: 120,
  sourceDir: RUNTIME_WORK_DIR,
});

const launchResult = await agentcoreRuntime.launch({ autoUpdateOnConflict: true });
console.log(`\n   Deployment initiated: ${JSON.stringify(launchResult)}`);

// Store for persistence across cells (Python: `%store launch_result`). The lab keeps its own
// config file beside the other labs' and mirrors it into the notebook state, so a restarted
// kernel can pick these values back up in Part 5.
const CONFIG_FILE = "environments/evaluation_lab_config.json";

type EvalLabConfig = {
  agent_name: string;
  agent_id: string;
  agent_arn: string;
  region: string;
  session_id: string;
  evaluator_id: string;
  online_config_id: string;
};

const labConfig: EvalLabConfig = {
  agent_name: AGENT_NAME,
  agent_id: launchResult.agentId,
  agent_arn: launchResult.agentArn,
  region,
  session_id: "",
  evaluator_id: "",
  online_config_id: "",
};

async function saveLabConfig(config: EvalLabConfig): Promise<void> {
  await writeFile(CONFIG_FILE, `${JSON.stringify(config, null, 2)}\n`);
  await state.set("evaluation_lab", config);
}

/** Python: `%store -r launch_result` and friends, reading the config file then the state file. */
async function loadLabConfig(): Promise<EvalLabConfig> {
  try {
    return JSON.parse(await Deno.readTextFile(CONFIG_FILE)) as EvalLabConfig;
  } catch {
    return await state.require<EvalLabConfig>("evaluation_lab", "the Part 2 cells above");
  }
}

await saveLabConfig(labConfig);

In [ ]:
// Wait for deployment to become READY
console.log("⏳ Waiting for agent to reach READY status...");
console.log("   (Check the AgentCore console for live status)\n");

const endStatuses = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"];

let status = "";
while (true) {
  const statusResponse = await agentcoreRuntime.status();
  status = (statusResponse.endpoint as { status?: string } | null)?.status ?? "UNKNOWN";
  console.log(`   Status: ${status}`);
  if (endStatuses.includes(status)) break;
  await sleep(15_000);
}

if (status === "READY") {
  console.log("\n✅ Agent deployed successfully!");
  console.log(`   Agent ID:  ${launchResult.agentId}`);
  console.log(`   Agent ARN: ${launchResult.agentArn}`);
} else {
  console.log(`\n❌ Deployment failed with status: ${status}`);
  console.log("   Check the AgentCore console for error details.");
}

In [ ]:
// Invoke the agent to generate traces
const sessionId = crypto.randomUUID();
console.log(`📡 Session ID: ${sessionId}`);
console.log("   (Save this — we use it for on-demand evaluations)\n");

// Three test invocations covering different scenarios
const testPrompts: [string, string][] = [
  ["Math (in-scope)", "How much is 2 + 2?"],
  ["Weather (in-scope)", "What is the weather like today?"],
  ["Capital (OUT-OF-SCOPE)", "Can you tell me the capital of the United States?"],
];

for (const [label, prompt] of testPrompts) {
  console.log(`   🔹 ${label}`);
  console.log(`      Prompt: ${prompt}`);
  const response = await agentcoreRuntime.invoke({ prompt }, { sessionId });
  const rendered = typeof response === "string" ? response : JSON.stringify(response);
  console.log(`      Response: ${rendered.slice(0, 120)}...`);
  console.log();
}

console.log(
  "\n✅ Agent invocations complete — traces are being generated in AgentCore Observability!",
);
console.log("   Waiting 30 seconds before starting the span-ingestion check...");
await sleep(30_000);
console.log("   ✅ Initial wait complete.");

// Store session_id for use in later cells (Python: `%store session_id`)
labConfig.session_id = sessionId;
await saveLabConfig(labConfig);

---
# Part 3: Explore Built-In Evaluators

AgentCore provides **13 pre-configured evaluators** ready to use immediately.
Let's explore them before running any evaluations.

In [ ]:
import { Evaluation } from "../toolkit/mod.ts";

const evalClient = new Evaluation({ region });

console.log("📋 All Available Built-In Evaluators:\n");

const available = await evalClient.listEvaluators();
const evaluators = available.evaluators ?? [];

// Group by inferred category
const categories: Record<string, [string, string][]> = {
  "Response Quality": [],
  "Task Completion": [],
  "Tool Level": [],
  "Safety": [],
};

for (const ev of evaluators) {
  const name = ev.evaluatorId ?? "";
  const desc = ev.description ?? "";
  if (name.includes("Goal")) {
    categories["Task Completion"].push([name, desc]);
  } else if (name.includes("Tool")) {
    categories["Tool Level"].push([name, desc]);
  } else if (["Harm", "Stereo", "Refusal", "Privacy", "Topic"].some((w) => name.includes(w))) {
    categories["Safety"].push([name, desc]);
  } else {
    categories["Response Quality"].push([name, desc]);
  }
}

for (const [category, items] of Object.entries(categories)) {
  console.log("─".repeat(50));
  console.log(`  ${category}`);
  console.log("─".repeat(50));
  for (const [name, desc] of items) {
    console.log(`  • ${name}`);
    console.log(`    ${desc.slice(0, 80)}`);
  }
  console.log();
}

console.log(`Total: ${evaluators.length} built-in evaluators`);

In [ ]:
// Deep-dive into Builtin.Correctness
console.log("🔍 Deep-dive: Builtin.Correctness\n");

const correctnessDetail = await evalClient.getEvaluator({ evaluatorId: "Builtin.Correctness" });

console.log(JSON.stringify(correctnessDetail, null, 2));

console.log();
console.log("💡 Key Observations:");
console.log("   • This is an LLM-as-judge evaluator");
console.log("   • It operates at TRACE level (one score per user turn)");
console.log("   • The configuration is fixed — you cannot modify it");
console.log("   • That fixed config ensures consistency across all evaluations");
console.log("   • Notice the rating scale — only 3 levels: Incorrect/Partially/Correct");
console.log("   ➡️  We'll create a custom 5-level version in Part 4");

---
# Part 4: Create a Custom Evaluator

Built-in `Correctness` only has 3 levels. We want a **5-level scale** that also penalizes out-of-scope answers.

Our custom evaluator will:
- Use **5 levels**: Very Good → Good → OK → Poor → Very Poor
- Penalize agents that answer questions **outside their scope** (weather + math only)
- Operate at **TRACE level** (per user turn)

In [ ]:
// Define the custom evaluator configuration
const customEvalConfig = {
  llmAsAJudge: {
    modelConfig: {
      bedrockEvaluatorModelConfig: {
        modelId: "global.anthropic.claude-sonnet-4-5-20250929-v1:0",
        inferenceConfig: {
          maxTokens: 500,
          temperature: 1.0,
        },
      },
    },
    instructions: "You are evaluating the quality of an AI assistant's response.\n" +
      "You are given a task and the assistant's candidate response.\n\n" +
      "Evaluate whether the response accurately addresses the question.\n" +
      "Consider factual accuracy, completeness, and clarity.\n\n" +
      "**IMPORTANT SCOPE RULE**: This assistant is ONLY supposed to answer " +
      "questions about weather and mathematical calculations. " +
      "If the assistant answers questions outside this scope " +
      "(e.g., geography, history, general knowledge), " +
      "it MUST receive a 'Very Poor' rating regardless of the response quality.\n\n" +
      "Context: {context}\n" +
      "Candidate Response: {assistant_turn}",
    ratingScale: {
      numerical: [
        {
          value: 1.0,
          label: "Very Good",
          definition: "Response is completely accurate, directly answers the question, " +
            "and stays within the agent's defined scope (weather or math).",
        },
        {
          value: 0.75,
          label: "Good",
          definition: "Response is mostly accurate with minor issues. Core answer is " +
            "correct. Stays within scope.",
        },
        {
          value: 0.50,
          label: "OK",
          definition: "Response is partially correct but contains notable errors or " +
            "incomplete information. Within scope but imperfect.",
        },
        {
          value: 0.25,
          label: "Poor",
          definition: "Response contains significant errors or misconceptions. " +
            "Mostly incorrect but within scope.",
        },
        {
          value: 0.0,
          label: "Very Poor",
          definition: "Response is completely incorrect, OR the agent answered a " +
            "question outside its defined scope (non-weather, non-math). " +
            "Scope violations always result in Very Poor regardless of answer quality.",
        },
      ],
    },
  },
};

console.log("📊 Custom Evaluator Configuration:");
console.log(
  `   Model: ${customEvalConfig.llmAsAJudge.modelConfig.bedrockEvaluatorModelConfig.modelId}`,
);
console.log("   Level: TRACE (one result per user turn)");
console.log("   Scale: 5 levels (0.0 Very Poor → 1.0 Very Good)");
console.log("   Scope rule: Out-of-scope answers → Very Poor");

In [ ]:
console.log("🔧 Creating custom evaluator...\n");

const customEvaluator = await evalClient.createEvaluator({
  name: "response_quality_with_scope",
  level: "TRACE",
  description: "5-level response quality evaluator that penalizes out-of-scope answers. " +
    "Built for weather+math agents.",
  config: customEvalConfig,
});

const evaluatorId = customEvaluator.evaluatorId ?? "";

console.log("✅ Custom Evaluator Created!");
console.log(`   Evaluator ID: ${evaluatorId}`);
console.log("   Name: response_quality_with_scope");
console.log("   Level: TRACE");
console.log("   Scale: Very Poor (0.0) → Very Good (1.0)");

// Store for later use (Python: `%store evaluator_id`)
labConfig.evaluator_id = evaluatorId;
await saveLabConfig(labConfig);

---
# Part 5: On-Demand Evaluations

Now let's evaluate the agent session we created in Part 2.

We'll run evaluations at all three levels:
- **Session level**: Goal Success Rate (whole conversation)
- **Trace level**: Correctness + our custom evaluator (per turn)
- **Span level**: Tool Selection + Parameter Accuracy (per tool call)

In [ ]:
import { ObservabilityClient } from "../toolkit/mod.ts";
import type { EvaluationResults, Span } from "../toolkit/mod.ts";

// Restore stored variables (Python: `%store -r launch_result` / `session_id` / `evaluator_id`)
const evalContext = await loadLabConfig();

console.log("📂 Loaded evaluation context:");
console.log(`   Agent ID:     ${evalContext.agent_id}`);
console.log(`   Agent ARN:    ${evalContext.agent_arn}`);
console.log(`   Session ID:   ${evalContext.session_id}`);
console.log(`   Custom Eval:  ${evalContext.evaluator_id}`);

// `display(Markdown(...))` has no Deno equivalent outside the kernel, so this renders markdown
// when the kernel is present and prints it when these cells run as a plain script.
const jupyter = (Deno as unknown as {
  jupyter?: {
    md: (strings: TemplateStringsArray, ...values: unknown[]) => unknown;
    display: (value: unknown) => Promise<void>;
  };
}).jupyter;

async function showMarkdown(text: string): Promise<void> {
  if (jupyter) {
    await jupyter.display(jupyter.md`${text}`);
  } else {
    console.log(text);
  }
}

/** Block until the session spans are visible in AgentCore Observability. */
async function waitForSessionSpans(
  agentId: string,
  sessionId: string,
  attempts = 12,
  delaySeconds = 20,
  lookbackDays = 1,
): Promise<Span[]> {
  const obsClient = new ObservabilityClient({ region });
  for (let attempt = 1; attempt <= attempts; attempt++) {
    const endTime = Date.now();
    const startTime = endTime - lookbackDays * 24 * 60 * 60 * 1000;
    const spans = await obsClient.querySpansBySession({
      sessionId,
      startTimeMs: startTime,
      endTimeMs: endTime,
      agentId,
    });
    if (spans.length > 0) {
      console.log(`   Found ${spans.length} spans for session ${sessionId}.`);
      return spans;
    }
    if (attempt === attempts) {
      throw new Error(
        `No spans found for session ${sessionId} after ${attempts} checks. ` +
          "Check CloudWatch / AgentCore Observability ingestion.",
      );
    }
    console.log(
      `   Spans not ready yet (attempt ${attempt}/${attempts}). Waiting ${delaySeconds}s...`,
    );
    await sleep(delaySeconds * 1000);
  }
  return [];
}

async function runEvaluationWithRetry(
  agentId: string,
  sessionId: string,
  evaluatorIds: string[],
): Promise<EvaluationResults> {
  await waitForSessionSpans(agentId, sessionId);
  return await evalClient.run({ agentId, sessionId, evaluators: evaluatorIds });
}

In [ ]:
console.log("🎯 SESSION-LEVEL: Goal Success Rate\n");
console.log("Question: Did the agent complete all the user's goals across the full conversation?");
console.log();

const goalResults = await runEvaluationWithRetry(
  evalContext.agent_id,
  evalContext.session_id,
  ["Builtin.GoalSuccessRate"],
);

console.log(`Results (${goalResults.results.length} result for the whole session):\n`);
for (const result of goalResults.results) {
  await showMarkdown(`
**Evaluator**: \`${result.evaluatorName}\`

**Score**: \`${result.label}\` (${result.value})

**Explanation**: ${result.explanation}

**Tokens Used**: ${JSON.stringify(result.tokenUsage)}
`);
  console.log("─".repeat(60));
}

In [ ]:
console.log("✅ TRACE-LEVEL: Correctness\n");
console.log("Question: Is each individual agent response factually correct?");
console.log("(One result per user turn)");
console.log();

const correctnessResults = await runEvaluationWithRetry(
  evalContext.agent_id,
  evalContext.session_id,
  ["Builtin.Correctness"],
);

console.log(`Results (${correctnessResults.results.length} results — one per turn):\n`);
for (const [index, result] of correctnessResults.results.entries()) {
  await showMarkdown(`
**Turn ${index + 1}** | **${result.evaluatorName}**

**Score**: \`${result.label}\` (${result.value})

**Explanation**: ${result.explanation}
`);
  console.log("─".repeat(60));
}

In [ ]:
console.log("🔧 SPAN-LEVEL: Tool Selection & Parameter Accuracy\n");
console.log("Question: Did the agent choose the right tools with the right parameters?");
console.log("(One result per tool call — multiple possible per turn)");
console.log();

const toolResults = await runEvaluationWithRetry(
  evalContext.agent_id,
  evalContext.session_id,
  [
    "Builtin.ToolSelectionAccuracy",
    "Builtin.ToolParameterAccuracy",
  ],
);

console.log(
  `Results (${toolResults.results.length} results — one per tool invocation per metric):\n`,
);
for (const result of toolResults.results) {
  await showMarkdown(`
**Metric**: \`${result.evaluatorName}\`

**Score**: \`${result.label}\` (${result.value})

**Explanation**: ${result.explanation}

**Context**: ${JSON.stringify(result.context).slice(0, 200)}
`);
  console.log("─".repeat(60));
}

In [ ]:
console.log("🎨 TRACE-LEVEL: Custom Evaluator (5-level + scope enforcement)\n");
console.log("Question: Is the response high quality AND does the agent stay in scope?");
console.log();
console.log("Expected behavior:");
console.log("  • Math turn (2+2)    → Very Good or Good (correct, in-scope)");
console.log("  • Weather turn       → Good or OK (tool used correctly, in-scope)");
console.log("  • Capital turn       → Very Poor (out-of-scope! capital is not weather or math)");
console.log();

const customResults = await runEvaluationWithRetry(
  evalContext.agent_id,
  evalContext.session_id,
  [evalContext.evaluator_id],
);

console.log(`Results (${customResults.results.length} results — one per turn):\n`);
for (const [index, result] of customResults.results.entries()) {
  const scoreValue = result.value ?? 0;
  const scoreText = result.value !== undefined ? String(result.value) : "n/a";
  const scoreEmoji = scoreValue >= 0.75 ? "✅" : (scoreValue >= 0.5 ? "⚠️" : "❌");
  await showMarkdown(`
${scoreEmoji} **Turn ${index + 1}** | \`${result.evaluatorName}\`

**Score**: \`${result.label}\` (${scoreText})

**Explanation**: ${result.explanation}
`);
  console.log("─".repeat(60));
}

console.log();
console.log("💡 Did the third turn get 'Very Poor'?");
console.log(
  "   That's our custom scope rule kicking in — the agent answered geography, not math/weather.",
);

In [ ]:
// Save evaluation results to a JSON file
console.log("💾 Saving evaluation results to file...\n");

const OUTPUT_FILE = "eval_results/lab_evaluation_output.json";
await Deno.mkdir("eval_results", { recursive: true });

const saveResults = await evalClient.run({
  agentId: evalContext.agent_id,
  sessionId: evalContext.session_id,
  evaluators: [
    "Builtin.GoalSuccessRate",
    "Builtin.Correctness",
    evalContext.evaluator_id,
  ],
  output: OUTPUT_FILE,
});

// Check file was saved
const savedText = await Deno.readTextFile(OUTPUT_FILE).catch(() => null);
if (savedText !== null) {
  const saved = JSON.parse(savedText) as { summary: { totalEvaluations: number } };
  console.log(`✅ Results saved to: ${OUTPUT_FILE}`);
  console.log(
    `   ${saved.summary.totalEvaluations} results in the file, ` +
      `${saveResults.results.length} returned by the API.`,
  );
} else {
  console.log("ℹ️  Results may have been saved to a different location.");
  console.log("   Check the evalClient.run() output above for the file path.");
}

---
# Part 6: Online Evaluation — Continuous Production Monitoring

Now let's configure **online evaluation** — this runs automatically against live traffic.

Online evaluation is the production monitoring mode. Once configured, it evaluates your agent continuously based on the sampling rate you define.

> **Note on timing**: online evaluation is asynchronous and can lag well behind the invocations that feed it — minutes, sometimes longer, and the lag grows when the runtime log group is still filling. Nothing in this part blocks the lab: if no results have appeared by the time you finish reading, run the cleanup cell anyway and rely on the on-demand results from Part 5.
>
> If results never appear, check, in this order:
> - **CloudWatch Transaction Search** is enabled in this account and region
> - the runtime log group `/aws/bedrock-agentcore/runtimes/<agent_id>-DEFAULT` exists and has recent OTEL log records
> - the online config's status is `ENABLED` (printed by the next cell) and its `evaluationExecutionRoleArn` role still exists
> - the invocations in this part really produced traces — the Part 5 span query is the quickest proof

In [ ]:
console.log("⚙️ Creating Online Evaluation Configuration...\n");

// Online evaluation is asynchronous and may never report within the lab: a failure here must not
// stop the notebook, so the cell reports and moves on to cleanup.
let onlineConfigId = "";
try {
  const onlineResponse = await evalClient.createOnlineConfig({
    agentId: evalContext.agent_id,
    configName: "pumping_code_quality_monitor",
    samplingRate: 100, // 100% for this lab; use 10-20% in production
    evaluatorList: [
      "Builtin.GoalSuccessRate",
      "Builtin.Correctness",
      "Builtin.ToolParameterAccuracy",
      "Builtin.ToolSelectionAccuracy",
      evalContext.evaluator_id, // our custom scope evaluator
    ],
    configDescription: "Continuous quality monitoring for weather+math assistant. " +
      "Checks correctness, tool accuracy, and scope adherence.",
    autoCreateExecutionRole: true,
  });

  onlineConfigId = onlineResponse.onlineEvaluationConfigId ?? "";

  console.log("✅ Online Evaluation Config Created!");
  console.log(`   Config ID: ${onlineConfigId}`);
  console.log("   Sampling Rate: 100% (all sessions)");
  console.log("   Evaluators: 5 (4 built-in + 1 custom)");
  console.log("   Status: ENABLED — evaluating live traffic automatically");

  evalContext.online_config_id = onlineConfigId;
  await saveLabConfig(evalContext);
} catch (e) {
  console.log(`⚠️  Could not create the online evaluation config: ${String(e).slice(0, 200)}`);
  console.log("   The on-demand results from Part 5 are unaffected — continue to the next cells.");
}

In [ ]:
// Verify the config is active
if (onlineConfigId) {
  const configDetails = await evalClient.getOnlineConfig({ configId: onlineConfigId });

  console.log("📋 Online Evaluation Config Details:");
  console.log(JSON.stringify(configDetails, null, 2));
} else {
  console.log("ℹ️  No online evaluation config to inspect — the cell above did not create one.");
}

In [ ]:
// Invoke the agent multiple times to trigger online evaluation
import {
  BedrockAgentCoreClient,
  InvokeAgentRuntimeCommand,
} from "@aws-sdk/client-bedrock-agentcore";

const agentcoreDataClient = new BedrockAgentCoreClient({ region });

/** Invoke the deployed agent directly, without the toolkit's session bookkeeping. */
async function invokeAgentDirect(agentArn: string, prompt: string): Promise<string> {
  const response = await agentcoreDataClient.send(
    new InvokeAgentRuntimeCommand({
      agentRuntimeArn: agentArn,
      qualifier: "DEFAULT",
      payload: new TextEncoder().encode(JSON.stringify({ prompt })),
      contentType: "application/json",
      accept: "application/json",
    }),
  );
  return (await response.response?.transformToString()) ?? "{}";
}

console.log("📡 Triggering new agent sessions to activate online evaluation...\n");
console.log("   These new sessions will be automatically evaluated by our online config.");
console.log();

const onlinePrompts = [
  "How much is 7 + 9 + 10 * 2?",
  "Is it raining right now?",
  "What is 20% of 300?",
  "What can you help me with?",
  "What is the capital of New York State?", // out-of-scope!
];

for (const prompt of onlinePrompts) {
  console.log(`   → ${prompt}`);
  try {
    await invokeAgentDirect(evalContext.agent_arn, prompt);
    console.log("     ✅ Response received");
  } catch (e) {
    console.log(`     ⚠️  ${String(e).slice(0, 60)}`);
  }
}

console.log();
console.log("✅ Agent invocations complete!");
console.log("   Online evaluation is now processing these sessions in the background.");
console.log("   Results appear in CloudWatch under AgentCore Observability.");
console.log();
console.log("📊 Where to view results:");
console.log("   AWS Console → CloudWatch → Gen AI Observability → AgentCore → Agents");
console.log("   → Select your agent → DEFAULT endpoint → Evaluation tab");
console.log("   (May take 2-5 minutes to appear)");

---
# 🧹 Cleanup

Clean up all resources to avoid ongoing costs.

**Note**: The deployed AgentCore Runtime agent will also stop incurring costs when idle (after the `idleTimeout` period).

In [ ]:
import { DeleteAgentRuntimeCommand } from "@aws-sdk/client-bedrock-agentcore-control";

console.log("🧹 Starting cleanup...\n");

// The Python lab re-reads `%store -r online_config_id` / `evaluator_id` / `launch_result` at each
// step; the lab's config file holds all three.
const cleanupContext = await loadLabConfig();

// ── Step 1: Delete online evaluation config ───────────────────────────────
try {
  console.log("Step 1: Deleting online evaluation config...");
  if (!cleanupContext.online_config_id) throw new Error("no online evaluation config was created");
  await evalClient.deleteOnlineConfig({ configId: cleanupContext.online_config_id });
  console.log(`   ✅ Online config deleted: ${cleanupContext.online_config_id}`);
} catch (e) {
  console.log(`   ⚠️  ${String(e).slice(0, 80)}`);
}

// ── Step 2: Delete custom evaluator ──────────────────────────────────────
try {
  console.log("\nStep 2: Deleting custom evaluator...");
  if (!cleanupContext.evaluator_id) throw new Error("no custom evaluator was created");
  await evalClient.deleteEvaluator({ evaluatorId: cleanupContext.evaluator_id });
  console.log(`   ✅ Custom evaluator deleted: ${cleanupContext.evaluator_id}`);
} catch (e) {
  console.log(`   ⚠️  ${String(e).slice(0, 80)}`);
}

// ── Step 3: Delete the deployed agent ────────────────────────────────────
try {
  console.log("\nStep 3: Deleting AgentCore Runtime agent...");
  const agentId = cleanupContext.agent_id;
  if (!agentId) throw new Error("cleanupContext.agent_id is missing");

  await ctrl.send(new DeleteAgentRuntimeCommand({ agentRuntimeId: agentId }));
  console.log(`   Delete requested for agent runtime: ${agentId}`);

  let stillThere: { status?: string } | undefined;
  for (let attempt = 1; attempt <= 20; attempt++) {
    const listed = await ctrl.send(new ListAgentRuntimesCommand({ maxResults: 100 }));
    stillThere = (listed.agentRuntimes ?? []).find((item) => item.agentRuntimeId === agentId);
    if (!stillThere) {
      console.log(`   ✅ Agent ${agentId} deleted`);
      break;
    }
    console.log(`   Waiting for deletion... current status: ${stillThere.status}`);
    await sleep(10_000);
  }
  if (stillThere) {
    console.log(`   ⚠️  Agent ${agentId} still exists after waiting. Status: ${stillThere.status}`);
  }
} catch (e) {
  console.log(`   ⚠️  ${String(e).slice(0, 120)}`);
  console.log("   You can also delete from the AgentCore console manually.");
}

// ── Step 4: Clean up local files ─────────────────────────────────────────
// The Dockerfile is generated by the toolkit's launch(); the stored runtime entry is this port's
// equivalent of the Python lab's `.bedrock_agentcore.yaml`.
for (const path of [AGENT_FILE, REQUIREMENTS_FILE, `${EVALUATION_DIR}/Dockerfile`]) {
  try {
    await Deno.remove(path);
  } catch {
    // Already gone.
  }
}
await clearStoredRuntimeConfig(AGENT_NAME);

console.log("\n🎉 Cleanup complete!");
console.log(
  "   eval_results/ folder retained — check lab_evaluation_output.json for your results.",
);

---
# 🎉 Lab Complete!

## What You Accomplished

| Step | Concept | What You Learned |
|------|---------|------------------|
| Part 2 | Deploy agent to Runtime | OTEL instrumentation is automatic |
| Part 3 | Explore built-in evaluators | 13 pre-built metrics, fixed configs |
| Part 4 | Custom evaluator (5-level) | LLM-as-judge with scope enforcement |
| Part 5 | On-demand: GoalSuccessRate | Session-level task completion |
| Part 5 | On-demand: Correctness | Trace-level per-turn accuracy |
| Part 5 | On-demand: Tool metrics | Span-level tool usage quality |
| Part 5 | On-demand: Custom | Scope penalization in action |
| Part 6 | Online evaluation | Continuous automatic monitoring |
| Part 7 | Results analysis | How to read and act on scores |

## Key Takeaways

- 📊 **Three levels**: Span (tool calls) → Trace (turns) → Session (full conversation)
- 🤖 **13 built-in evaluators** cover most use cases immediately
- 🎨 **Custom evaluators** enable domain-specific and scope-aware quality checks
- 💡 **The explanation field** is the most valuable field — read it to understand WHY
- 🔄 **On-demand + Online** complement each other: dev-time testing + prod monitoring